================================================================================
模块 02：市场微观结构与订单簿动力学 · 第 3/5 课
第 3 课：订单流不平衡 (Order Flow Imbalance, OFI) 与高频价格预测 (参考答案版)
================================================================================

【Q1 投决会深度解析】
为什么 OFI 相比于单纯的“主动买入成交量减主动卖出成交量”（净主动买入额 Net Taker Volume），能够提供信噪比更高、更纯净的短期价格预测能力？请从做市商被动撤单与盘口新挂单事件的信息含量角度推导：
- **[被动挂撤单信息含量与状态完全捕捉]**：
  - **[1. Net Taker Volume 的信息截断]**：主动成交量（Net Taker Volume）仅记录了最终撮合落地的历史事实，而忽略了挂单簿供给侧的动态响应。如果一位知情大买家正在进场，敏锐的做市商会立刻在卖方撤单（减少卖盘供给）并在买方主动垫高买价挂单；在这个过程中，即使尚未发生大额主动撮合成交，价格向上的净压力已经形成。
  - **[2. 撤单事件的隐蔽意图]**：被动撤单（Cancellation）蕴含着极其高浓度的私人信息。例如卖一档出现大规模撤单，通常意味着原本提供流动性的空头做市商认输出逃，上方阻力瞬间被抽空。单纯看主动成交量指标对此类“供给消失引发的暴涨”完全无知无觉，而 OFI 通过 $I^a_t$ 严格计入此类撤单通量。
  - **[3. 净通量对冲的代数优越性]**：OFI 将新到挂单（Maker 流）、市价吃单（Taker 流）与主动撤单（Cancel 流）这三种驱动盘口变动的离散事件全部正交映射在同一物理代数天平上，因此能以更纯粹的信噪比领先预测价格下一跳跳跃。

【Q2 投决会深度解析】
为什么在实盘高频量化交易中，单步 OFI 对价格变化的解释力在毫秒至秒级极强（R^2 常常达到 50%~70%），但随着预测时间窗口拉长（如 5分钟、15分钟），其预测能力呈现指数级衰减？做市商如何利用 OFI 衰减半衰期设定库存管理风控？
- **[流动性补充动力学与库存半衰期控制]**：
  - **[1. 微观机械冲击的局部性]**：单步 OFI 衡量的是瞬时盘口供需失衡对盘口当前挂单深度的机械性消耗。在极短时间尺度（毫秒至 1~2 秒），由于外部补充挂单尚未到达交易所网络队列，盘口被迫发生确定性价格跳跃以吸纳冲击，因此瞬时线性解释力极高。
  - **[2. 做市商回补与均值回归]**：在几秒钟之后，随着跨市场套利算法、统计套利模型与流动性提供者的新挂单迅速回填被吃穿的深度，盘口流动性重新充裕，甚至产生反向弹性均值回归（Mean Reversion）。原本的微观冲击被宏观大单与跨资产关联流稀释，OFI 的预测残差随时间呈指数级发散衰减。
  - **[3. 做市商库存半衰期风控设定]**：做市商在被动吃下客户订单后持有单边敞口，依据 OFI 衰减模型：
    - 若高频信号显示当前 OFI 处于前 1 秒的强冲击窗口，算法应坚决停止向该方向逆势挂单；
    - 根据经验衰减半衰期 $t_{1/2} = \ln 2 / \lambda_{\text{decay}}$（通常在 1.5~3 秒），在半衰期过后，原冲击动力已消退 50% 以上，做市商方可启动向公允中间价双向挂单以平抑库存、赚取回归利差。

【Q3 投决会深度解析】
在跨多档盘口（Level 1 至 Level 10）拓展 Multi-Level OFI 时，为什么越深的档位（如买五、买十）需要赋予越大的权重衰减（如指数衰减 exp(-alpha * k)）？深档挂单中存在大量“虚假挂单（Ghost Orders / Spoofing）”会对未加权的多档 OFI 造成怎样的污染？
- **[距离衰减与虚假挂单过滤拓扑]**：
  - **[1. 成交概率随盘口距离几何衰减]**：依据排队撮合物理法则，挂在买十档位的限价单要被撮合成交，必须先由市价单连续吃穿前面买一至买九档的所有深度。因此越深档位的订单，其在当前瞬间被成交的条件概率越趋近于零，其对当前跳跃的价格指示意义微乎其微。
  - **[2. 虚假挂单（Spoofing）的重灾区]**：恶意操纵算法（Spoofing Algos）极度热衷于在买四、买五或更深档位批量挂出上万手虚假大单，以此人为拉高该侧的静态深度，诱骗高频跟风算法。由于这些挂单距离最优价较远，几乎没有意外被市价单真实成交的被动风险，操纵者可以在毫秒内安全撤销。
  - **[3. 衰减权重消除伪信号]**：如果采用简单算术平均计算多档 OFI，深档虚假大单的剧烈挂撤将产生虚假的巨额 OFI 脉冲，彻底淹没买一档真实的微观流动性通量。引入指数衰减权重 $w_k = \exp(-\alpha (k-1))$，使三档以外的权重迅速归零，既吸收了浅层盘口弹性，又在数理上免疫了深档虚假幽灵单的噪音污染。


In [ ]:
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd


def calculate_ofi(
    b_price_prev: float,
    b_size_prev: int,
    b_price_curr: float,
    b_size_curr: int,
    a_price_prev: float,
    a_size_prev: int,
    a_price_curr: float,
    a_size_curr: int,
) -> float:
    """
    计算单个 Tick 步长的 Level 1 订单流不平衡 (OFI):
        OFI = I_Bid - I_Ask
    """
    if b_price_curr > b_price_prev:
        i_bid = b_size_curr
    elif b_price_curr == b_price_prev:
        i_bid = b_size_curr - b_size_prev
    else:
        i_bid = -b_size_prev

    if a_price_curr < a_price_prev:
        i_ask = a_size_curr
    elif a_price_curr == a_price_prev:
        i_ask = a_size_curr - a_size_prev
    else:
        i_ask = -a_size_prev

    return float(i_bid - i_ask)


def calculate_tick_ofi_series(tick_df: pd.DataFrame) -> pd.Series:
    """
    从连续的 Tick 快照数据集中批量计算单步 OFI 序列
    """
    n = len(tick_df)
    ofi_vals = [0.0]

    for t in range(1, n):
        row_prev = tick_df.iloc[t - 1]
        row_curr = tick_df.iloc[t]
        val = calculate_ofi(
            row_prev["b_price"], int(row_prev["b_size"]),
            row_curr["b_price"], int(row_curr["b_size"]),
            row_prev["a_price"], int(row_prev["a_size"]),
            row_curr["a_price"], int(row_curr["a_size"]),
        )
        ofi_vals.append(val)

    return pd.Series(ofi_vals, index=tick_df.index)


def calculate_multi_level_ofi(
    b_prices_prev: List[float],
    b_sizes_prev: List[int],
    b_prices_curr: List[float],
    b_sizes_curr: List[int],
    a_prices_prev: List[float],
    a_sizes_prev: List[int],
    a_prices_curr: List[float],
    a_sizes_curr: List[int],
    decay: float = 0.5,
) -> float:
    """
    计算跨多档深度 (Multi-Level) 加权订单流不平衡
    """
    n_levels = min(len(b_prices_prev), len(b_prices_curr), len(a_prices_prev), len(a_prices_curr))
    total_ofi = 0.0

    for k in range(n_levels):
        w_k = np.exp(-decay * k)
        ofi_k = calculate_ofi(
            b_prices_prev[k], b_sizes_prev[k],
            b_prices_curr[k], b_sizes_curr[k],
            a_prices_prev[k], a_sizes_prev[k],
            a_prices_curr[k], a_sizes_curr[k],
        )
        total_ofi += w_k * ofi_k

    return float(total_ofi)


def estimate_ofi_price_impact(ofi_series: pd.Series, mid_price_series: pd.Series) -> Dict[str, float]:
    """
    估计 OFI 对中间价变化的一阶价格冲击系数与预测拟合度
    """
    d_mid = mid_price_series.diff()
    df = pd.DataFrame({"ofi": ofi_series, "d_mid": d_mid}).dropna()

    corr = float(df["ofi"].corr(df["d_mid"]))
    var_ofi = float(np.var(df["ofi"].values, ddof=1))
    if var_ofi > 1e-12:
        cov = float(np.cov(df["ofi"].values, df["d_mid"].values)[0, 1])
        beta = cov / var_ofi
    else:
        beta = 0.0

    r_squared = float(corr ** 2)

    return {
        "correlation": float(corr),
        "beta": float(beta),
        "r_squared": float(r_squared),
    }


if __name__ == "__main__":
    # ---------------- 基础测试 1: 基础单步 OFI 计算 ----------------
    ofi = calculate_ofi(10.0, 80, 10.1, 100, 10.2, 50, 10.2, 40)
    assert abs(ofi - 110.0) < 1e-4, f"Expected 110.0, got {ofi}"

    ofi2 = calculate_ofi(10.1, 100, 10.0, 70, 10.2, 50, 10.15, 60)
    assert abs(ofi2 - (-160.0)) < 1e-4, f"Expected -160.0, got {ofi2}"

    # ---------------- 基础测试 2: 多档 Multi-Level OFI ----------------
    bp_prev = [10.0, 9.9]
    bs_prev = [100, 200]
    bp_curr = [10.0, 9.9]
    bs_curr = [150, 200]

    ap_prev = [10.1, 10.2]
    as_prev = [80, 150]
    ap_curr = [10.1, 10.2]
    as_curr = [60, 150]

    multi_ofi = calculate_multi_level_ofi(
        bp_prev, bs_prev, bp_curr, bs_curr,
        ap_prev, as_prev, ap_curr, as_curr,
        decay=0.5,
    )
    assert abs(multi_ofi - 70.0) < 1e-4, f"Expected 70.0, got {multi_ofi}"

    # ---------------- 基础测试 3: OFI 时序与价格冲击回归 ----------------
    np.random.seed(42)
    n_ticks = 100
    synth_ofi = np.random.normal(0.0, 50.0, n_ticks)
    synth_d_mid = 0.002 * synth_ofi + np.random.normal(0.0, 0.02, n_ticks)
    synth_mid = 100.0 + np.cumsum(synth_d_mid)

    s_ofi = pd.Series(synth_ofi)
    s_mid = pd.Series(synth_mid)

    impact = estimate_ofi_price_impact(s_ofi, s_mid)
    assert impact["correlation"] > 0.85, f"OFI 与价格变化应高度正相关, 实际: {impact['correlation']}"
    assert 0.0015 <= impact["beta"] <= 0.0025, f"Beta 冲击系数应接近 0.002, 实际: {impact['beta']}"
    assert impact["r_squared"] > 0.70, f"R^2 应高于 0.70, 实际: {impact['r_squared']}"

    print("Lesson 03 Answer Verified!")
